# BirdCLEF 2024: Official Submission Notebook
This notebook runs 100% offline (no pip installs!) to comply with Kaggle's Code Competition rules. It extracts raw probabilities from Perch v2 and BirdNET TFLite models, maps them to the 182 competition species, and generates the `submission.csv`.

In [ ]:
# --- SUBMISSION WEIGHTS ---
# Change these weights to get your different scores!
# Perch Only: PERCH = 1.0, BIRDNET = 0.0
# BirdNET Only: PERCH = 0.0, BIRDNET = 1.0
# Ensemble: PERCH = 0.5, BIRDNET = 0.5
WEIGHT_PERCH = 0
WEIGHT_BIRDNET = 1

In [ ]:
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from tqdm.auto import tqdm

INPUT_DIR = Path('/kaggle/input')

# Dynamically locate the BirdCLEF dataset directory
try:
    COMP_DIR = next(INPUT_DIR.rglob('train_metadata.csv')).parent
    print(f"Found competition directory at: {COMP_DIR}")
except StopIteration:
    print("ERROR: Could not find train_metadata.csv. Did you attach the birdclef-2024 dataset?")
    COMP_DIR = INPUT_DIR / 'birdclef-2024' # fallback

TEST_AUDIO_DIR = COMP_DIR / 'test_soundscapes'

# If the test directory is empty (e.g., when editing interactively), use unlabeled soundscapes to test the code.
test_files = list(TEST_AUDIO_DIR.glob('*.ogg'))
if len(test_files) == 0:
    print("Hidden test set not found. Using 2 unlabeled soundscapes for local testing.")
    TEST_AUDIO_DIR = COMP_DIR / 'unlabeled_soundscapes'
    test_files = list(TEST_AUDIO_DIR.glob('*.ogg'))[:2]
else:
    print(f"Found {len(test_files)} hidden test files.")

# Load BirdCLEF metadata to get the 182 target species
train_df = pd.read_csv(COMP_DIR / 'train_metadata.csv')
TARGET_SPECIES = sorted(train_df['primary_label'].unique())
print(f"Target species count: {len(TARGET_SPECIES)}")

# Load taxonomy to map scientific names to species codes
taxa_df = pd.read_csv(COMP_DIR / 'eBird_Taxonomy_v2021.csv')
sci2code = dict(zip(taxa_df['SCI_NAME'].str.lower(), taxa_df['SPECIES_CODE']))


In [ ]:
# --- Initialize BirdNET (TFLite) ---
birdnet_idx_to_target = {}
if WEIGHT_BIRDNET > 0:
    try:
        tflite_path = next(INPUT_DIR.rglob("*.tflite"))
        try:
            labels_path = next(INPUT_DIR.rglob("*Labels.txt"))
        except StopIteration:
            labels_path = next(INPUT_DIR.rglob("*labels.txt"))
            
        print(f"Found BirdNET model at: {tflite_path}")
        
        interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
        interpreter.allocate_tensors()
        b_input_idx = interpreter.get_input_details()[0]['index']
        b_output_idx = interpreter.get_output_details()[0]['index']

        with open(labels_path, 'r') as f:
            birdnet_raw_labels = f.readlines()
            
        birdnet_sci_names = [line.split('_')[0].strip().lower() for line in birdnet_raw_labels]
        for idx, sci in enumerate(birdnet_sci_names):
            code = sci2code.get(sci)
            if code in TARGET_SPECIES:
                birdnet_idx_to_target[idx] = TARGET_SPECIES.index(code)
        print("BirdNET initialized.")
    except Exception as e:
        print(f"ERROR Initializing BirdNET: {e}")

# --- Initialize Perch v2 ---
perch_idx_to_target = {}
if WEIGHT_PERCH > 0:
    try:
        pb_path = next(INPUT_DIR.rglob("saved_model.pb"))
        perch_path = pb_path.parent
        _raw_model = tf.saved_model.load(str(perch_path))
        perch = _raw_model.signatures["serving_default"]
        
        # Dynamically find the Perch label CSV
        try:
            perch_label_file = next(INPUT_DIR.rglob("label.csv"))
        except StopIteration:
            perch_label_file = next(INPUT_DIR.rglob("labels.csv"))
            
        print(f"Found Perch label file at: {perch_label_file}")
        perch_label_csv = pd.read_csv(perch_label_file)
        
        # Handle variations in the label.csv column names
        code_col = 'ebird2021' if 'ebird2021' in perch_label_csv.columns else 'species_code'
        
        for idx, row in perch_label_csv.iterrows():
            code = row.get(code_col)
            if code in TARGET_SPECIES:
                perch_idx_to_target[idx] = TARGET_SPECIES.index(code)
        print("Perch v2 initialized.")
    except Exception as e:
        print(f"ERROR Initializing Perch: {e}")
        perch = None


In [ ]:
all_preds = []

for file_path in tqdm(test_files):
    audio_id = file_path.stem
    
    y_32k = np.array([])
    y_48k = np.array([])
    
    if WEIGHT_PERCH > 0:
        y_32k, _ = librosa.load(file_path, sr=32000, mono=True)
    if WEIGHT_BIRDNET > 0:
        y_48k, _ = librosa.load(file_path, sr=48000, mono=True)
        
    # Test files are exactly 240 seconds long, evaluated every 5 seconds
    # If audio is shorter, we handle it dynamically.
    duration = int(len(y_32k) / 32000) if len(y_32k) > 0 else int(len(y_48k) / 48000)
    if duration == 0: duration = 240 # Fallback
    
    for time_end in range(5, min(duration + 1, 241), 5):
        row_id = f"{audio_id}_{time_end}"
        final_probs = np.zeros(len(TARGET_SPECIES))
        
        # --- PERCH INFERENCE ---
        if WEIGHT_PERCH > 0 and perch is not None:
            start_idx = (time_end - 5) * 32000
            end_idx = time_end * 32000
            chunk = y_32k[start_idx:end_idx]
            if len(chunk) < 160000:
                chunk = np.pad(chunk, (0, 160000 - len(chunk)))
                
            x = tf.constant(chunk[None, :], dtype=tf.float32)
            perch_out = perch(inputs=x)["label"].numpy()[0]
            # Convert logits to probabilities via sigmoid
            perch_probs = 1 / (1 + np.exp(-perch_out))
            
            p_probs_mapped = np.zeros(len(TARGET_SPECIES))
            for p_idx, t_idx in perch_idx_to_target.items():
                p_probs_mapped[t_idx] = max(p_probs_mapped[t_idx], perch_probs[p_idx])
                
            final_probs += p_probs_mapped * WEIGHT_PERCH
            
        # --- BIRDNET INFERENCE ---
        if WEIGHT_BIRDNET > 0:
            # BirdNET uses 3s windows. For a 5s window, we evaluate two 3s windows and take the max.
            start1 = (time_end - 5) * 48000
            end1 = start1 + 144000
            start2 = (time_end - 3) * 48000
            end2 = start2 + 144000
            
            b_probs_mapped = np.zeros(len(TARGET_SPECIES))
            
            for s, e in [(start1, end1), (start2, end2)]:
                if e <= len(y_48k):
                    chunk = y_48k[s:e]
                    if len(chunk) < 144000:
                        chunk = np.pad(chunk, (0, 144000 - len(chunk)))
                    interpreter.set_tensor(b_input_idx, np.float32(chunk)[np.newaxis, :])
                    interpreter.invoke()
                    b_out = interpreter.get_tensor(b_output_idx)[0]
                    # BirdNET outputs probabilities directly
                    for b_idx, t_idx in birdnet_idx_to_target.items():
                        b_probs_mapped[t_idx] = max(b_probs_mapped[t_idx], b_out[b_idx])
                        
            final_probs += b_probs_mapped * WEIGHT_BIRDNET
            
        row_dict = {'row_id': row_id}
        for idx, sp in enumerate(TARGET_SPECIES):
            row_dict[sp] = final_probs[idx]
        all_preds.append(row_dict)

sub_df = pd.DataFrame(all_preds)
sub_df.to_csv('submission.csv', index=False)
print("Submission saved successfully! Shape:", sub_df.shape)

In [ ]:
sub_df.head(3)